In [ ]:
!pip install pandas numpy matplotlib scikit-learn transformers torch prophet
import os
print("Libraries installed and ready!")

# ***Cell 1 Imports & Device Setup***

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ***Cell 2 Fetch Dataset from Drive***

In [ ]:
from google.colab import drive
import glob, zipfile

drive.mount("/content/drive")

zip_matches = glob.glob("/content/drive/MyDrive/**/cleaned_dataset.zip", recursive=True)
if not zip_matches:
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive")

with zipfile.ZipFile(zip_matches[0], "r") as zf:
    zf.extractall("/content")

print(f"Extracted: {zip_matches[0]}")

Mounted at /content/drive
Extracted: /content/drive/MyDrive/cleaned_dataset.zip


# ***Cell 3 Load Sequence Data***

In [ ]:
data_path = "/content/dataset_v1_frozen/dataset_v1_cell_splits.npz"
if os.path.exists(data_path):
    data = np.load(data_path)
    X_train, y_train = data['X_train'], data['y_train_soh']
    X_val, y_val = data['X_val'], data['y_val_soh']
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], -1)
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], -1)
else:
    X_train = np.random.randn(800, 10, 153).astype(np.float32)
    y_train = np.linspace(1.0, 0.7, 800).astype(np.float32)
    X_val = np.random.randn(200, 10, 153).astype(np.float32)
    y_val = np.linspace(1.0, 0.7, 200).astype(np.float32)

# ***Cell 4 Build DataLoaders***

In [ ]:
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=32, shuffle=False)

# ***Cell 5 Define Tunable GRU Model***

In [ ]:
class TunableGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, output_dim=1):
        super(TunableGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.fc1(out[:, -1, :])
        out = self.relu(self.drop(out))
        return self.fc2(out).squeeze()

# ***Cell 6 Grid Search Space***

In [ ]:
search_space = {
    'hidden_dim': [32, 64],
    'num_layers': [1, 2],
    'dropout': [0.1, 0.3],
    'lr': [0.001, 0.005]
}

keys, values = zip(*search_space.items())
experiment_configs = [dict(zip(keys, v)) for v in itertools.product(*values)]

# ***Cell 7 Run Grid Search***

In [ ]:
experiment_logs = []
best_val_loss = float('inf')
best_config = None
best_model_state = None
epochs = 20
input_dim = X_train.shape[2]

for idx, config in enumerate(experiment_configs):
    model = TunableGRU(
        input_dim=input_dim,
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        dropout=config['dropout']
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)

    val_loss /= len(val_loader.dataset)

    config_log = config.copy()
    config_log['Experiment_ID'] = idx + 1
    config_log['Validation_MSE'] = val_loss
    experiment_logs.append(config_log)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_config = config
        best_model_state = model.state_dict()

# ***Cell 8 Save Best Model & Logs***

In [ ]:
df_logs = pd.DataFrame(experiment_logs)
df_logs = df_logs[['Experiment_ID', 'hidden_dim', 'num_layers', 'dropout', 'lr', 'Validation_MSE']]
df_logs = df_logs.sort_values(by='Validation_MSE').reset_index(drop=True)

torch.save(best_model_state, "/content/best_gru_tuned.pth")

print("Grid Search Complete. Best Model Saved to /content/best_gru_tuned.pth\n")
print(f"Best Configuration Found: {best_config}")
print(f"Best Validation MSE: {best_val_loss:.6f}\n")
print("Experiment Logs:")
display(df_logs)

Grid Search Complete. Best Model Saved to /content/best_gru_tuned.pth

Best Configuration Found: {'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.005}
Best Validation MSE: 0.009146

Experiment Logs:


,Experiment_ID,hidden_dim,num_layers,dropout,lr,Validation_MSE
0,14,64,2,0.1,0.005,0.009146
1,16,64,2,0.3,0.005,0.009402
2,15,64,2,0.3,0.001,0.009969
3,12,64,1,0.3,0.005,0.010217
4,6,32,2,0.1,0.005,0.010409
5,5,32,2,0.1,0.001,0.011089
6,4,32,1,0.3,0.005,0.011202
7,2,32,1,0.1,0.005,0.011246
8,8,32,2,0.3,0.005,0.011353
9,13,64,2,0.1,0.001,0.011617
